In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ultrasound_nerve_segmentation_path = kagglehub.competition_download('ultrasound-nerve-segmentation')

print('Data source import complete.')


In [ ]:
import os
import cv2
import timm
import glob
import torch
import random
import imagehash
import numpy as np
import pandas as pd
import torch.nn as nn
import albumentations as A
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

TRAIN_IMG_DIR = '/kaggle/input/ultrasound-nerve-segmentation/train'
TEST_IMG_DIR = '/kaggle/input/ultrasound-nerve-segmentation/test'

In [ ]:
def create_metadata_advanced(root_dir):
    mask_files = sorted(glob.glob(os.path.join(root_dir, "*_mask.tif")))
    data = []
    print("Đang quét và phân tích Metadata...")
    for mask_path in tqdm(mask_files):
        img_path = mask_path.replace("_mask.tif", ".tif")
        base_name = os.path.basename(img_path)
        subject_id = int(base_name.split("_")[0])
        mask = cv2.imread(mask_path, 0)
        mask_area = np.sum(mask > 0)
        has_nerve = 1 if mask_area > 0 else 0

        data.append({
            "img_path": img_path,
            "mask_path": mask_path,
            "subject_id": subject_id,
            "has_nerve": has_nerve,
            "mask_area": mask_area
        })
    df = pd.DataFrame(data)
    return df

def filter_duplicates(df):
    print(f"Số lượng ảnh ban đầu: {len(df)}")
    hashes = []
    print("Đang tính toán Image Hashes...")
    for path in tqdm(df['img_path']):
        img = Image.open(path)
        img_hash = imagehash.phash(img)
        hashes.append(str(img_hash))

    df['img_hash'] = hashes
    duplicate_groups = df[df.duplicated(subset=['img_hash'], keep=False)].groupby('img_hash')
    ids_to_drop = []
    print(f"Tìm thấy {len(duplicate_groups)} nhóm ảnh trùng lặp.")

    for img_hash, group in duplicate_groups:
        group_sorted = group.sort_values(by=['has_nerve', 'mask_area'], ascending=[False, False])
        survivor_idx = group_sorted.index[0]
        drop_indices = group_sorted.index[1:].tolist()
        ids_to_drop.extend(drop_indices)

    df_clean = df.drop(ids_to_drop).reset_index(drop=True)

    print(f"Đã xóa {len(ids_to_drop)} ảnh trùng lặp/mâu thuẫn.")
    print(f"Số lượng ảnh sau khi lọc: {len(df_clean)}")

    return df_clean

def get_sampler(df):
    """
    Tạo WeightedRandomSampler để cân bằng 50/50 giữa 2 class
    """
    class_counts = df['has_nerve'].value_counts().sort_index()
    weight_class0 = 1.0 / class_counts[0]
    weight_class1 = 1.0 / class_counts[1]
    samples_weights = df['has_nerve'].map({0: weight_class0, 1: weight_class1}).tolist()
    sampler = WeightedRandomSampler(
        weights=samples_weights,
        num_samples=len(df),
        replacement=True
    )
    return sampler

In [ ]:
class SegDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['img_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(row['mask_path'], 0)
        mask[mask > 0] = 1.0

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return image, mask

class ClsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['img_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = torch.tensor([row['has_nerve']], dtype=torch.float32)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']
        return image, label

train_transform = A.Compose(
    [
        A.Affine(scale=(0.8, 1.2), translate_percent=(-0.0625, 0.0625), rotate=(-15, 15)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        A.OneOf([
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.5),
            A.GridDistortion(p=0.5),
            A.OpticalDistortion(distort_limit=1, p=0.5),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.03), p=0.5),
            A.RandomBrightnessContrast(p=0.5),
             A.CLAHE(clip_limit=4.0),
        ], p=0.5),
        ToTensorV2(),
    ],
)

test_transform = A.Compose([
    A.Normalize(mean=(0.5, 0.5, 0.5),std=(0.5, 0.5, 0.5)),
    ToTensorV2(),
])

In [ ]:
#U-net and Dice loss define
class Conv_Block(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(Conv_Block, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, kernel_size=3, padding=1),
            nn.InstanceNorm2d(out_channel, affine=True),
            nn.LeakyReLU(0.1, inplace=False),
            nn.Conv2d(out_channel, out_channel, kernel_size=3, padding=1),
            nn.InstanceNorm2d(out_channel, affine=True),nn.ReLU(inplace=True),
            nn.LeakyReLU(0.1, inplace=False),
        )

    def forward(self, x):
        return self.layers(x)

class CBAM(nn.Module):
    """Convolution Block Attention Module"""
    def __init__(self, in_channel,ratio=16):
        super(CBAM, self).__init__()
        self.in_channel = in_channel
        self.MLP = nn.Sequential(
            nn.Linear(in_channel, in_channel // ratio, bias=False),
            nn.LeakyReLU(0.1, inplace=True), # Thay ReLU
            nn.Linear(in_channel // ratio, in_channel, bias=False)
        )
        self.Conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
    def forward(self, x):
        #channel attention
        pre_mlp1 = F.adaptive_avg_pool2d(x, (1,1))
        pre_mlp2 = F.adaptive_max_pool2d(x, (1,1))

        pre_mlp1 = pre_mlp1.view(pre_mlp1.size(0), -1)
        pre_mlp2 = pre_mlp2.view(pre_mlp2.size(0), -1)

        out_channel_attention = F.sigmoid(self.MLP(pre_mlp1) + self.MLP(pre_mlp2))

        x = x * out_channel_attention.view(x.size(0), x.size(1), 1, 1)

        #spatial attention
        pre_conv1,_ = torch.max(x, dim=1, keepdim=True)
        pre_conv2 = torch.mean(x, dim=1, keepdim=True)
        pre_conv = torch.cat([pre_conv1,pre_conv2],1)
        out_spatial_attention = F.sigmoid(self.Conv(pre_conv))

        x = x * out_spatial_attention
        return x

class GABM(nn.Module):
    """
        Global Attention Block Module
        Replace normal skip connection
    """
    def __init__(self, in_channel1,in_channel2, out_channel):
        super(GABM, self).__init__()
        self.conv1 = nn.Conv2d(in_channel1, out_channel, kernel_size=1)
        self.norm1 = nn.InstanceNorm2d(out_channel, affine=True)

        self.conv2 = nn.Conv2d(in_channel2, out_channel, kernel_size=1)
        self.norm2 = nn.InstanceNorm2d(out_channel, affine=True)

        self.conv3 = nn.Conv2d(out_channel, 1, kernel_size=1)
        self.norm3 = nn.InstanceNorm2d(1, affine=True)

        self.relu = nn.LeakyReLU(0.1, inplace=False)
    def forward(self, xl, xg):
        x1 = self.norm1(self.conv1(xl))
        if xg.size()[2:] != xl.size()[2:]:
            xg = F.interpolate(xg, size=xl.size()[2:], mode='bilinear', align_corners=True)
        x2 = self.norm2(self.conv2(xg))
        prex3 = self.relu(x1 + x2)
        x3 = torch.sigmoid(self.norm3(self.conv3(prex3)))
        out = xl * x3
        return out

class RGCM(nn.Module):
    """
        Residual Group Convolution Module
    """
    def __init__(self, in_channel, out_channel,groups=8, reduction=2):
        super(RGCM, self).__init__()
        self.diff_channel = False
        mid_channel = in_channel // reduction
        if mid_channel % groups != 0:
            mid_channel = ((mid_channel // groups) + 1) * groups

        self.conv1 = nn.Conv2d(in_channel, mid_channel, kernel_size=1, bias=False)
        self.bn1 = nn.InstanceNorm2d(mid_channel, affine=True)

        self.conv2 = nn.Conv2d(mid_channel, mid_channel, kernel_size=3,
                               stride=1, padding=1, groups=groups, bias=False)
        self.bn2 = nn.InstanceNorm2d(mid_channel, affine=True)

        self.conv3 = nn.Conv2d(mid_channel, out_channel, kernel_size=1, bias=False)
        self.bn3 = nn.InstanceNorm2d(out_channel, affine=True)

        self.shortcut = nn.Sequential()
        if in_channel != out_channel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channel, out_channel, kernel_size=1, bias=False),
                nn.InstanceNorm2d(out_channel, affine=True)
            )
        self.relu = nn.LeakyReLU(0.1, inplace=False)
    def forward(self, x):
        residual = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += residual
        out = self.relu(out)

        return out

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPP, self).__init__()
        # 1. Conv 1x1
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=False),
        )
        # 2. Dilated Convs (rate 6, 12, 18)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=False),
        )
        # 3. Global Avg Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=False),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(out_channels * 4, out_channels, 1, bias=False), # Concatenate 4 nhánh
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=False),
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        x3 = self.conv3(x)

        x4 = self.avg_pool(x)
        x4 = self.conv4(x4)
        x4 = F.interpolate(x4, size=x.size()[2:], mode='bilinear', align_corners=True)

        x_cat = torch.cat([x1, x2, x3, x4], dim=1)
        return self.final_conv(x_cat)

class ARGA_U_net(nn.Module):
    def __init__(self, in_channel, num_classes):
        super(ARGA_U_net, self).__init__()
        self.conv_in = Conv_Block(in_channel, 64)
        self.RCGM1 = RGCM(64, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.RCGM2 = RGCM(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.RCGM3 = RGCM(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.RCGM4 = RGCM(256, 512)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.RCGM5 = RGCM(512,1024)
        self.ASPP = ASPP(1024,1024)

        self.up5 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.GABM6 = GABM(512,1024,1024)
        self.dec_conv6 = nn.Sequential(
            nn.Conv2d(1024 + 512, 512, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(512, affine=True),
            nn.LeakyReLU(0.1, inplace=False),
            RGCM(512, 512),
            CBAM(512)
        )
        self.up6 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.GABM7 = GABM(256,512,512)
        self.dec_conv7 = nn.Sequential(
            nn.Conv2d(512 + 256, 256, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(256, affine=True),
            nn.LeakyReLU(0.1, inplace=False),
            RGCM(256, 256),
            CBAM(256)
        )

        self.up7 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.GABM8 = GABM(128,256,256)
        self.dec_conv8 = nn.Sequential(
            nn.Conv2d(256 + 128, 128, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(128, affine=True),
            nn.LeakyReLU(0.1, inplace=False),
            RGCM(128, 128),
            CBAM(128)
        )

        self.up8 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.GABM9 = GABM(64,128,128)
        self.dec_conv9 = nn.Sequential(
            nn.Conv2d(128 + 64, 64, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(64, affine=True),
            nn.LeakyReLU(0.1, inplace=False),
            RGCM(64, 64),
            CBAM(64)
        )
        self.ds_head1 = nn.Conv2d(128, num_classes, kernel_size=1)
        self.ds_head2 = nn.Conv2d(256, num_classes, kernel_size=1)
        self.ds_head3 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.last_layer = nn.Conv2d(64, num_classes, kernel_size=1)
    def forward(self, x):
        #encoder
        x = self.conv_in(x)
        x1 = self.RCGM1(x)
        x2 = self.RCGM2(self.pool1(x1))
        x3 = self.RCGM3(self.pool2(x2))
        x4 = self.RCGM4(self.pool3(x3))
        #bottleneck
        x5 = self.ASPP(self.RCGM5(self.pool4(x4)))

        #decoder
        x5_up = self.up5(x5)
        x4g = self.GABM6(xl=x4, xg=x5)
        if x5_up.size() != x4g.size():
             x5_up = F.interpolate(x5_up, size=x4g.shape[2:], mode='bilinear', align_corners=True)

        x6 = self.dec_conv6(torch.cat([x5_up, x4g], dim=1))

        x6_up = self.up6(x6)
        x3g = self.GABM7(xl=x3, xg=x6)
        if x6_up.size() != x3g.size():
            x6_up = F.interpolate(x6_up, size=x3.shape[2:], mode='bilinear', align_corners=True)
        x7 = self.dec_conv7(torch.cat([x6_up, x3g], dim=1))

        x7_up = self.up7(x7)
        x2g = self.GABM8(xl=x2, xg=x7)
        if x7.size() != x2g.size():
            x7_up = F.interpolate(x7_up, size=x2g.shape[2:], mode='bilinear', align_corners=True)
        x8 = self.dec_conv8(torch.cat([x7_up, x2g], dim=1))

        x8_up = self.up8(x8)
        x1g = self.GABM9(xl=x1, xg=x8)
        if x8.size() != x1g.size():
            x8_up = F.interpolate(x8_up, size=x1g.shape[2:], mode='bilinear', align_corners=True)
        x9 = self.dec_conv9(torch.cat([x8_up, x1g], dim=1))

        out = self.last_layer(x9)
        if self.training:
            out1 = self.ds_head1(x6)
            out2 = self.ds_head2(x7)
            out3 = self.ds_head3(x8)
            return [out1, out2, out3, out_final]
        else:
            return out_final

def calculate_dice_score(preds, targets, smooth=1e-5, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)

    intersection = (preds_flat * targets_flat).sum()
    dice = (2. * intersection + smooth) / (preds_flat.sum() + targets_flat.sum() + smooth)
    return dice.item()

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

def compute_cls_metrics_sklearn(preds, targets, probs=None):
    """
    Tính toán metrics sử dụng Sklearn.
    Args:
        preds (Tensor): Binary predictions (0 hoặc 1). Shape: (Batch_size, 1) hoặc (Batch_size,)
        targets (Tensor): Ground truth labels (0 hoặc 1). Shape tương tự preds.
        probs (Tensor, optional): Xác suất (Probabilities) chưa qua threshold. Dùng để tính ROC-AUC.
    Returns:
        acc, prec, rec, f1 (và auc nếu có probs)
    """
    y_pred = preds.detach().cpu().numpy().flatten()
    y_true = targets.detach().cpu().numpy().flatten()
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    metrics = {
        'acc': acc,
        'prec': prec,
        'rec': rec,
        'f1': f1
    }
    if probs is not None:
        y_prob = probs.detach().cpu().numpy().flatten()
        try:
            if len(np.unique(y_true)) > 1:
                auc = roc_auc_score(y_true, y_prob)
                metrics['auc'] = auc
            else:
                metrics['auc'] = 0.5
        except ValueError:
            metrics['auc'] = 0.0

    return metrics

class FocalLoss(nn.Module):
    def __init__(self):
        super(FocalLoss, self).__init__()
    def forward(self, pred, target, alpha=0.70, gamma=2):
        target = target.float()
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = alpha * target + (1 - alpha) * (1 - target)
        F_loss = alpha_t * (1-pt)**gamma * bce
        focal_loss = F_loss.mean()
        return focal_loss

class DiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super(DiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target, smooth=1.):
        target = target.float()
        bce_loss = self.bce(pred, target)
        pred_soft = torch.sigmoid(pred)

        pred_flat = pred_soft.view(-1)
        target_flat = target.view(-1)

        intersection = (pred_flat * target_flat).sum()
        dice_score = (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
        dice_loss = 1 - dice_score

        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss

class DeepSupervisionLoss(nn.Module):
    def __init__(self, base_loss_fn, weights=[0.3, 0.4, 0.5, 1.0]):
        super().__init__()
        self.base_loss_fn = base_loss_fn
        self.weights = weights

    def forward(self, outputs, target):
        if isinstance(outputs, list):
            loss = 0
            for output, weight in zip(outputs, self.weights):
                if output.shape[-2:] != target.shape[-2:]:
                    output = F.interpolate(output, size=target.shape[-2:], mode='bilinear', align_corners=False)

                loss += weight * self.base_loss_fn(output, target)
            return loss
        else:
            return self.base_loss_fn(outputs, target)

def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler, task_type='seg'):
    """
    Hàm train chung cho cả 2 task.
    Args:
        task_type (str): 'seg' (Segmentation) hoặc 'cls' (Classification)
    """
    model.train()
    epoch_loss = 0

    pbar = tqdm(loader, desc=f"Training {task_type.upper()}", leave=False)

    for i, (images, targets) in enumerate(pbar):
        images = images.to(device)
        targets = targets.to(device)

        if task_type == 'seg':
            if targets.ndim == 3:
                targets = targets.unsqueeze(1)
            targets = targets.float()

        elif task_type == 'cls':
            if targets.ndim == 1:
                targets = targets.unsqueeze(1)
            targets = targets.float()

        with autocast(dtype=torch.float16):
            outputs = model(images)
            loss = loss_fn(outputs, targets)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    return epoch_loss / len(loader)

def validate_one_epoch(model, loader, loss_fn, device, task_type='seg'):
    """
    Hàm validate chung cho cả 2 task.
    Trả về: Loss trung bình và Dictionary chứa các Metrics.
    """
    model.eval()
    total_loss = 0

    metrics = {
        'dice': [],
        'iou': [],
        'acc': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'auc': []
    }
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Validating {task_type.upper()}", leave=False)

        for images, targets in pbar:
            images = images.to(device)
            targets = targets.to(device)
            if task_type == 'seg':
                if targets.ndim == 3: targets = targets.unsqueeze(1)
                targets = targets.float()
            elif task_type == 'cls':
                if targets.ndim == 1: targets = targets.unsqueeze(1)
                targets = targets.float()
            outputs = model(images)

            if isinstance(outputs, list):
                final_output = outputs[-1]
            else:
                final_output = outputs
            loss = loss_fn(final_output, targets)
            total_loss += loss.item()
            probs = torch.sigmoid(final_output)
            preds = (probs > 0.5).float()

            if task_type == 'seg':
                batch_dice = calculate_dice_score(preds, targets)
                metrics['dice'].extend(batch_dice)
                metrics['iou'].extend(batch_dice / (2-batch_dice))
            elif task_type == 'cls':
                batch_metrics = compute_cls_metrics_sklearn(preds, targets, probs)

                metrics['acc'].append(batch_metrics['acc'])
                metrics['precision'].append(batch_metrics['prec'])
                metrics['recall'].append(batch_metrics['rec'])
                metrics['f1'].append(batch_metrics['f1'])
                if 'auc' in batch_metrics:
                    metrics['auc'].append(batch_metrics['auc'])

            pbar.set_postfix({'val_loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(loader)

    results = {'loss': avg_loss}

    if task_type == 'seg':
        results['dice'] = np.mean(metrics['dice'])
        results['iou'] = np.mean(metrics['iou'])
        print(f" >> Val Dice: {results['dice']:.4f}")
    elif task_type == 'cls':
        results['acc'] = np.mean(metrics['acc'])
        results['precision'] = np.mean(metrics['precision'])
        results['recall'] = np.mean(metrics['recall'])
        results['f1'] = np.mean(metrics['f1'])
        results['auc'] = np.mean(metrics['auc'])
        print(f" >> Val Acc: {results['acc']:.4f} | Prec: {results['precision']:.4f}\n| Rec: {results['recall']:.4f} | F1: {results['f1']:.4f}")
        print(f" >> AUC: {results['auc']:.4f}")

    return results

def run_training_task(task_type, model, loader, optimizer, loss_fn, device, scaler):
    """
    Wrapper để chạy train cho 1 task cụ thể.
    """
    loss = train_one_epoch(
        model=model,
        loader=loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=device,
        scaler=scaler,
        task_type=task_type
    )
    return loss

def run_validation_task(task_type, model, loader, loss_fn, device):
    """
    Wrapper để chạy validate.
    """
    metrics = validate_one_epoch(
        model=model,
        loader=loader,
        loss_fn=loss_fn,
        device=device,
        task_type=task_type
    )
    return metrics

In [ ]:
df = create_metadata_advanced(TRAIN_IMG_DIR)
df = filter_duplicates(df)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df['has_nerve'], groups=df['subject_id'])):
    df.loc[val_idx, 'fold'] = fold

cls_train_df = df[df['fold'] != 0].reset_index(drop=True)
cls_val_df   = df[df['fold'] == 0].reset_index(drop=True)
cls_sampler = get_sampler(cls_train_df)

seg_train_df = df[(df['fold'] != 0) & (df['has_nerve'] == 1)]
seg_train_df = seg_train_df[seg_train_df['mask_area'] > 50].reset_index(drop=True)

seg_val_df   = df[(df['fold'] == 0) & (df['has_nerve'] == 1)].reset_index(drop=True)

print(f"Cls Train: {len(cls_train_df)} | Cls Val: {len(cls_val_df)}")
print(f"Seg Train: {len(seg_train_df)} | Seg Val: {len(seg_val_df)}")

seg_train_ds = SegDataset(seg_train_df, transform=train_transform)
seg_train_loader = DataLoader(seg_train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

seg_val_ds = SegDataset(seg_val_df, transform=test_transform)
seg_val_loader = DataLoader(seg_val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

cls_train_ds = ClsDataset(cls_train_df, transform=train_transform)
cls_train_loader = DataLoader(cls_train_ds, batch_size=16, sampler=cls_sampler,
                              shuffle=False, num_workers=2, pin_memory=True)

cls_val_ds = ClsDataset(cls_val_df, transform=test_transform)
cls_val_loader = DataLoader(cls_val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
#setup env
device_seg = torch.device("cuda:0")
device_cls = torch.device("cuda:1")
seg_model = ARGA_U_net(in_channel=3, num_classes=1).to(device_seg)
cls_model = timm.create_model('tf_efficientnet_b0_ns', pretrained=True, num_classes=1).to(device_cls)
print(f"Bộ nhớ GPU 0: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
print(f"Bộ nhớ GPU 1: {torch.cuda.memory_allocated(1) / 1024**2:.2f} MB")

scaler_seg = GradScaler('cuda')
scaler_cls = GradScaler('cuda')

optimizer_seg = optim.AdamW(seg_model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer_cls = optim.Adam(cls_model.parameters(), lr=1e-4, weight_decay=1e-4)

loss_fn_seg = DeepSupervisionLoss()
loss_fn_cls = FocalLoss()

In [ ]:
#Training
NUM_EPOCHS = 30
best_dice = 0.0
best_acc = 0.0

print(f"Bắt đầu Training song song trên: {device_seg} và {device_cls}")
for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*10} EPOCH {epoch+1}/{NUM_EPOCHS} {'='*10}")
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_seg = executor.submit(
            run_training_task,
            task_type='seg',
            model=seg_model,
            loader=seg_train_loader,
            optimizer=optimizer_seg,
            loss_fn=loss_fn_seg,
            device=device_seg,
            scaler=scaler_seg
        )

        future_cls = executor.submit(
            run_training_task,
            task_type='cls',
            model=cls_model,
            loader=cls_train_loader,
            optimizer=optimizer_cls,
            loss_fn=loss_fn_cls,
            device=device_cls,
            scaler=scaler_cls
        )

        loss_seg = future_seg.result()
        loss_cls = future_cls.result()

    print(f"[Train Done] Seg Loss: {loss_seg:.4f} | Cls Loss: {loss_cls:.4f}")
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_val_seg = executor.submit(
            run_validation_task, 'seg', seg_model, seg_val_loader, loss_fn_seg, device_seg
        )
        future_val_cls = executor.submit(
            run_validation_task, 'cls', cls_model, cls_val_loader, loss_fn_cls, device_cls
        )

        val_res_seg = future_val_seg.result()
        val_res_cls = future_val_cls.result()
    print(f"[SEG Val] Loss: {val_res_seg['loss']:.4f} | Dice: {val_res_seg['dice']:.4f}")
    if val_res_seg['dice'] > best_dice:
        best_dice = val_res_seg['dice']
        torch.save(seg_model.state_dict(), "best_seg_model.pth")
        print("Saved Best SEG Model!")
    print(f"[CLS Val] Loss: {val_res_cls['loss']:.4f} | Acc: {val_res_cls['acc']:.4f} | F1: {val_res_cls['f1']:.4f}")
    if val_res_cls['acc'] > best_acc:
        best_acc = val_res_cls['acc']
        torch.save(cls_model.state_dict(), "best_cls_model.pth")
        print("Saved Best CLS Model!")

In [ ]:
def find_best_seg_threshold(model, loader, device, step=0.05):
    model.eval()
    all_probs = []
    all_masks = []

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Inference Seg"):
            images = images.to(device)
            outputs = model(images)
            if isinstance(outputs, list):
                out_seg = outputs[-1]
            else:
                out_seg = outputs
            probs = torch.sigmoid(out_seg)
            all_probs.append(probs.cpu())
            all_masks.append(masks.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_masks = torch.cat(all_masks, dim=0)

    if all_masks.ndim == 3: all_masks = all_masks.unsqueeze(1)
    all_masks = (all_masks > 0).float()
    best_thres = 0.5
    best_dice = 0.0
    thresholds = np.arange(0.1, 0.96, step)

    for threshold in thresholds:
        preds = (all_probs > threshold).float()

        intersection = (preds * all_masks).sum(dim=(1, 2, 3))
        union = preds.sum(dim=(1, 2, 3)) + all_masks.sum(dim=(1, 2, 3))
        dice_scores = (2. * intersection + 1e-6) / (union + 1e-6)
        mean_dice = dice_scores.mean().item()

        print(f"{threshold:<10.2f} | {mean_dice:<10.4f}")

        if mean_dice > best_dice:
            best_dice = mean_dice
            best_thres = threshold
    return best_thres

def find_best_cls_threshold(model, loader, device, step=0.05):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Inference Cls"):
            images = images.to(device)
            labels = labels.float().cpu().numpy().flatten()

            outputs = model(images)
            if isinstance(outputs, list):
                outputs = outputs[-1]
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()

            all_probs.extend(probs)
            all_labels.extend(labels)

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    best_f1 = 0
    best_cls_thres = 0.5
    thresholds = np.arange(0.1, 0.96, step)

    for thres in thresholds:
        preds = (all_probs > thres).astype(int)

        f1 = f1_score(all_labels, preds, zero_division=0)
        acc = accuracy_score(all_labels, preds)
        rec = recall_score(all_labels, preds, zero_division=0)
        prec = precision_score(all_labels, preds, zero_division=0)

        print(f"{thres:<10.2f} | {f1:<10.4f} | {acc:<10.4f} | {rec:<10.4f} | {prec:<10.4f}")
        if f1 > best_f1:
            best_f1 = f1
            best_cls_thres = thres
    if best_cls_thres < 0.2:
        best_cls_thres = 0.5
    elif best_cls_thres > 0.9:
         best_cls_thres = 0.8
    return best_cls_thres

In [ ]:
def find_thresholds_parallel(seg_model, seg_loader, device_seg,
                             cls_model, cls_loader, device_cls):
    print(f"Bắt đầu tìm ngưỡng song song...")
    with ThreadPoolExecutor(max_workers=2) as executor:
        print(f"   -> Đẩy tác vụ Seg sang {device_seg}")
        future_seg = executor.submit(
            find_best_seg_threshold,
            model=seg_model,
            loader=seg_loader,
            device=device_seg,
            step=0.05
        )
        print(f"   -> Đẩy tác vụ Cls sang {device_cls}")
        future_cls = executor.submit(
            find_best_cls_threshold,
            model=cls_model,
            loader=cls_loader,
            device=device_cls,
            step=0.05
        )

        best_seg_thres = future_seg.result()
        best_cls_thres = future_cls.result()
    return best_seg_thres, best_cls_thres

best_seg, best_cls = find_thresholds_parallel(
    seg_model, seg_val_loader, device_seg,
    cls_model, cls_val_loader, device_cls
)
print(f"   Seg Threshold: {best_seg}")
print(f"   Cls Threshold: {best_cls}")

if (best_seg > 0.85) or (best_seg < 0.15):
    best_seg = 0.5
if best_cls < 0.2:
    best_cls = 0.5
elif best_cls > 0.9:
    best_cls = 0.8

In [ ]:
def rle_encode(mask):
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def remove_small_objects(mask, min_size=100):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_size:
            cleaned_mask[labels == i] = 1
    return cleaned_mask

def keep_largest_component(mask):
    mask = mask.astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels < 2:
        return mask
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    cleaned_mask = np.zeros_like(mask)
    cleaned_mask[labels == largest_label] = 1
    return cleaned_mask

def fill_holes(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(mask)
    cv2.drawContours(filled_mask, contours, -1, 1, thickness=cv2.FILLED)

    return filled_mask

def inference_pipeline(img_path, seg_model, cls_model, device_seg, device_cls):
    """
    Pipeline xử lý cho 1 ảnh đơn lẻ.
    """
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    augmented = test_transform(image=image)
    img_tensor = augmented['image'].unsqueeze(0)
    cls_model.eval()
    with torch.no_grad():
        img_cls = img_tensor.to(device_cls)
        cls_logits = cls_model(img_cls)
        if isinstance(cls_logits, list): cls_logits = cls_logits[-1]

        nerve_prob = torch.sigmoid(cls_logits).item()
    if nerve_prob < 0.15:
        return np.zeros(img_cls, dtype=np.uint8), nerve_prob

    seg_model.eval()
    with torch.no_grad():
        img_seg = img_tensor.to(device_seg)
        seg_out = seg_model(img_seg)
        if isinstance(seg_out, list): seg_out = seg_out[-1]

        mask_prob = torch.sigmoid(seg_out).squeeze().cpu().numpy()

    mask_binary = (mask_prob > best_seg).astype(np.uint8)

    mask_clean = remove_small_objects(mask_binary, min_size=100)
    mask_clean = keep_largest_component(mask_clean)
    mask_clean = fill_holes(mask_clean)
    mask_area = np.sum(mask_clean)
    final_mask = mask_clean

    if nerve_prob < 0.55:
        if mask_area < 2000:
            final_mask = np.zeros_like(mask_clean)
    elif nerve_prob < 0.8:
        if mask_area < 500:
            final_mask = np.zeros_like(mask_clean)
    else:
        pass

    return final_mask, nerve_prob

In [ ]:
DEVICE_SEG = torch.device('cuda:0')
DEVICE_CLS = torch.device('cuda:1')
test_files = glob.glob(os.path.join("/kaggle/input/ultrasound-nerve-segmentation/test", "*.tif"))
results = []

print("Bắt đầu Inference...")

for img_path in tqdm(test_files):
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    pred_mask, prob = inference_pipeline(
        img_path, seg_model, cls_model, DEVICE_SEG, DEVICE_CLS
    )

    if np.sum(pred_mask) > 0:
        rle = rle_encode(pred_mask)
    else:
        rle = ""

    results.append({
        "img": img_id,
        "pixels": rle
    })
results = sorted(results, key=lambda x: x['img'])
submission_df = pd.DataFrame(results)
submission_df.to_csv("submission.csv", index=False)